<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_2_Feature_Scaling/Day_11_Log_BoxCox_and_Power_Transformations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 11: Log, Box-Cox, and Power Transformations

## Phase 2 | Month 5 | Week 2 | Day 11

**🎯 Goal:** Reduce skewness and make distributions more Gaussian using mathematical transformations.
**⏱️ Study Session:** ~2 hours

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import PowerTransformer
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
N = 1000
# Highly right-skewed income data (typical in Kenya)
income = np.random.lognormal(10.5, 1.3, N)

transforms = {
    'Original':  income,
    'Log':       np.log1p(income),
    'Sqrt':      np.sqrt(income),
    'Box-Cox':   stats.boxcox(income)[0],
    'Yeo-Johnson': PowerTransformer(method='yeo-johnson').fit_transform(income.reshape(-1,1)).ravel(),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, vals) in zip(axes, transforms.items()):
    ax.hist(vals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    sk = stats.skew(vals)
    ax.set_title(f'{name}\nskewness={sk:.2f}', fontweight='bold', fontsize=9)
plt.tight_layout()
fig.savefig('/tmp/day11_transforms.png', dpi=120)
print('Saved /tmp/day11_transforms.png')
plt.close()

## When to Apply Which Transformation

| Transformation | When to Use | Limitations |
|----------------|-------------|-------------|
| `log(x)` / `log1p(x)` | Right-skewed, all positive values | Cannot use if x ≤ 0 |
| `sqrt(x)` | Mild right skew | Cannot use if x < 0 |
| **Box-Cox** | Optimises λ for normality | Requires x > 0 |
| **Yeo-Johnson** | Like Box-Cox but handles x ≤ 0 | Slightly slower |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 800
# Predict M-Pesa float balance from transaction volume
tx_vol = np.random.lognormal(5, 1.2, N)     # right-skewed predictor
float_bal = np.log(tx_vol) * 2000 + np.random.normal(0, 500, N)

X = tx_vol.reshape(-1, 1)
y = float_bal

raw_r2  = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2').mean()
yj_pipe = Pipeline([('yj', PowerTransformer()), ('lr', LinearRegression())])
yj_r2   = cross_val_score(yj_pipe, X, y, cv=5, scoring='r2').mean()

print(f'R² without transformation: {raw_r2:.4f}')
print(f'R² with Yeo-Johnson:       {yj_r2:.4f}')

## 💪 Exercises

### Exercise 1
Apply Box-Cox and Yeo-Johnson to the `annual_claims_kes` column from the NHIF dataset. Plot before and after. Which gives a more Gaussian result?

### Exercise 2
Fit a Linear Regression predicting NHIF claims from age and income. Compare R² and residual normality with and without log-transforming claims.

In [ ]:
# Your code here


## 📋 Summary

- ✓ Log transform is the most common fix for right-skewed positive data
- ✓ Box-Cox finds optimal λ — requires x > 0
- ✓ Yeo-Johnson works on all values including negatives and zeros
- ✓ Sklearn's `PowerTransformer` implements both in pipelines

## 🚀 Next Steps
**Day 12: Binning and Discretisation**